# Naive Bayes

**Objetivo:** usar o Naive Bayes gaussiano no Iris (vendo as curvas normais por classe) e o multinomial num pequeno problema de texto, notando a velocidade. Probabilidade condicional explícita, sem caixa-preta.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Naive Bayes gaussiano no Iris

O `GaussianNB` estima, para cada classe, a **média** e o **desvio** de cada característica — supondo uma normal. Depois combina pela regra de Bayes.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=SEMENTE, stratify=y)
modelo = GaussianNB().fit(X_tr, y_tr)
print("acuracia no teste:", round(modelo.score(X_te, y_te), 3))
print("medias por classe (uma linha por classe):")
print(np.round(modelo.theta_, 2))

## 2. As curvas normais estimadas

Para uma característica (comprimento da pétala), desenhamos a normal que o modelo ajustou a cada classe. Onde as curvas se cruzam fica a zona de confusão entre as espécies.

In [ ]:
caracteristica = 2   # comprimento da petala
grade = np.linspace(X[:, caracteristica].min()-0.5, X[:, caracteristica].max()+0.5, 300)

figura = go.Figure()
cores = [AZUL, VERDE, VERMELHO]
for c in range(3):
    media = modelo.theta_[c, caracteristica]
    desvio = np.sqrt(modelo.var_[c, caracteristica])
    densidade = np.exp(-0.5 * ((grade - media) / desvio) ** 2) / (desvio * np.sqrt(2*np.pi))
    figura.add_trace(go.Scatter(x=grade, y=densidade, mode="lines",
                                line=dict(color=cores[c]), name=iris.target_names[c]))
figura.update_layout(title="Curvas normais por classe (comprimento da petala)",
                     xaxis_title="cm", yaxis_title="densidade", height=360,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 3. Naive Bayes multinomial para texto

Em texto, cada documento vira um vetor de **contagens de palavras** (`CountVectorizer`) e o `MultinomialNB` classifica. Usamos um mini-corpus embutido (frases de esporte × tecnologia) — sem downloads.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

textos = [
    "o time venceu o jogo e marcou tres gols",
    "o jogador foi campeao do torneio de futebol",
    "a torcida comemorou a vitoria no estadio",
    "o atacante marcou o gol da vitoria no jogo",
    "o novo processador e a placa de video sao rapidos",
    "o software roda no computador com muita memoria",
    "o aplicativo usa inteligencia artificial e dados",
    "a rede neural treina no processador da maquina",
]
rotulos = ["esporte", "esporte", "esporte", "esporte",
           "tecnologia", "tecnologia", "tecnologia", "tecnologia"]

vetorizador = CountVectorizer()
X_texto = vetorizador.fit_transform(textos)
classificador = MultinomialNB().fit(X_texto, rotulos)
print("vocabulario (", len(vetorizador.get_feature_names_out()), "palavras)")

novas = ["o time marcou um gol no jogo", "a maquina usa inteligencia artificial"]
previsto = classificador.predict(vetorizador.transform(novas))
for frase, classe in zip(novas, previsto):
    print("->", classe.ljust(11), "|", frase)

## Exercício

O modelo classificou as duas frases novas corretamente mesmo sem nunca ter visto exatamente essas combinações de palavras. Por que a suposição de independência entre palavras não estragou a classificação?

<details><summary>Ver resposta</summary>

Porque para **decidir a classe** basta que a soma (em log) das evidências de cada palavra aponte para o lado certo — não é preciso que a probabilidade estimada seja exata. Palavras como "gol"/"jogo" empurram forte para "esporte" e "inteligencia"/"maquina" para "tecnologia"; mesmo tratando as palavras como independentes (o que ignora que "inteligencia" e "artificial" andam juntas), a **ordem** entre as duas classes se mantém e a decisão acerta.

</details>